# 🎙️ [실습] 5명이 '사과'라고 말하면? 소리 지문으로 목소리 비교하기!
> **수업 대상**: 코딩을 처음 배우는 고등학교 1학년 (초심자 환영!)
> **준비물**: 구글 코랩(Google Colab), 마이크(없으면 음성 파일 업로드로도 가능!)
> **선수 지식**: 1차시의 파형·주파수 개념 (코사인 유사도는 아래에서 다시 설명해요!)
---
### 📋 이 실습에서 배우는 것 (3줄 요약)
1. 🎤 5명이 각자 마이크로 **'사과'**라고 1~2초 말하기
2. 🔢 각 목소리를 5개 숫자로 요약한 **소리 지문(특징 벡터)** 뽑아내기
3. 📐 **코사인 유사도**(두 화살표가 같은 방향인지 재는 방법)로 누구 목소리가 닮았는지 비교하기
---
### 🚀 실행 순서
1. 아래 코드 셀을 **위에서부터 순서대로** 실행하세요 (▶ 버튼 클릭!)
2. 마지막 셀을 실행하면 Gradio 웹 화면이 열립니다
3. 각자 마이크로 '사과'를 말하고 **[분석]** 버튼을 눌러보세요!

> ✅ **체크포인트**: Gradio 화면에 5개의 녹음 칸이 떴다 → 성공! 이제 목소리를 모아볼까요? 🎉


In [ ]:
!pip -q install soundfile gradio

In [ ]:
#한글을 그래프에 보여주는 라이브러리
!pip install koreanize_matplotlib
import koreanize_matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 38.5 MB/s eta 0:00:00


In [ ]:
import numpy as np

def make_feature_from_wave(y, sr):
    # 스테레오 → 모노
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)
    # 정규화
    peak = np.max(np.abs(y)) + 1e-12
    y = (y/peak)*0.98

    # ---- (A) 전체 FFT (창 적용) ----
    N = len(y)
    if N < 2048:
        # 너무 짧으면 패딩
        y = np.pad(y, (0, 2048-N))
        N = len(y)
    win = np.hanning(N).astype(np.float32)
    X = np.fft.rfft(y*win)
    freqs = np.fft.rfftfreq(N, 1/sr)
    mag = np.abs(X)

    # ---- (B) 간단 특징들 ----
    # 1) RMS (에너지)
    rms = float(np.sqrt(np.mean(y**2) + 1e-12))
    # 2) ZCR (거칠기/무성음 힌트)
    zcr = float(np.mean(np.abs(np.diff(np.sign(y)))>0))

    # 3) 대역 에너지(저/중/고)
    def band_energy(f_low, f_high):
        idx = np.logical_and(freqs>=f_low, freqs<f_high)
        return float(np.sum(mag[idx]**2))
    lowE  = band_energy(   0,  300)   # 저역(기본주기)
    midE  = band_energy( 300, 3000)   # 중역(모음 포만트)
    highE = band_energy(3000, 8000)   # 고역(마찰음/자음)

    feat = np.array([rms, zcr, lowE, midE, highE], dtype=np.float32)
    feat /= (np.linalg.norm(feat) + 1e-12)  # 정규화(벡터 길이=1 가깝게)
    return feat, freqs, mag


In [ ]:
# --- 필요한 패키지 ---
import numpy as np
import gradio as gr
import tempfile
import matplotlib.pyplot as plt

# --- 음성 -> 특징벡터 (간단 버전) ---
def make_feature_from_wave(y, sr):
    # y: 소리 파형(숫자들의 나열), sr: 1초에 몇 번 기록했는지(샘플링 레이트)
    # 돌려주는 값: 5개 숫자(RMS, ZCR, 저/중/고 대역 에너지)로 요약한 "소리 지문" 벡터
    # 스테레오 -> 모노
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)
    # 정규화
    peak = np.max(np.abs(y)) + 1e-12
    y = (y/peak)*0.98

    # FFT
    N = len(y)
    if N < 2048:
        y = np.pad(y, (0, 2048-N))
        N = len(y)
    win = np.hanning(N).astype(np.float32)
    X = np.fft.rfft(y*win)
    freqs = np.fft.rfftfreq(N, 1/sr)
    mag = np.abs(X)

    # 특징 5가지를 뽑아요 (소리의 지문!)
    # 1) RMS: 소리의 평균적인 크기(에너지)
    # 2) ZCR: 파형이 0을 얼마나 자주 가로지르는지(자음의 거친 정도 힌트)
    # 3) 저/중/고 대역 에너지: 낮은~높은 소리가 각각 얼마나 섞였는지
    rms = float(np.sqrt(np.mean(y**2) + 1e-12))
    zcr = float(np.mean(np.abs(np.diff(np.sign(y))) > 0))

    def band_energy(f_low, f_high):
        idx = np.logical_and(freqs >= f_low, freqs < f_high)
        return float(np.sum(mag[idx]**2))
    lowE  = band_energy(   0,  300)
    midE  = band_energy( 300, 3000)
    highE = band_energy(3000, 8000)

    feat = np.array([rms, zcr, lowE, midE, highE], dtype=np.float32)
    feat /= (np.linalg.norm(feat) + 1e-12)  # 벡터 길이를 1로 맞춰주기(정규화: 비교하기 쉽게!)
    return feat, freqs, mag

# --- 유틸 ---
def cosine_sim(a,b):
    # 두 소리 지문 벡터가 같은 방향을 볼수록 1(100%)에 가까워져요!
    return float(np.dot(a,b) / ((np.linalg.norm(a)+1e-12)*(np.linalg.norm(b)+1e-12)))

def plot_spectrum(freqs, mag, fmax=8000):
    plt.figure(figsize=(6,3))
    plt.plot(freqs, 20*np.log10(mag+1e-12))
    plt.xlim(0, fmax)
    plt.xlabel("Frequency (Hz)"); plt.ylabel("Magnitude (dB)")
    plt.title("FFT Spectrum")
    plt.tight_layout()
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    plt.savefig(tmp.name, dpi=150); plt.close()
    return tmp.name  # 파일 경로 반환

def plot_similarity_matrix(mat, labels):
    plt.figure(figsize=(4.5,4))
    plt.imshow(mat, vmin=0, vmax=1, origin="upper")
    plt.xticks(range(len(labels)), labels)
    plt.yticks(range(len(labels)), labels)
    plt.colorbar(label="Cosine similarity (↑=more similar)")
    plt.title("Cosine Similarity")
    plt.tight_layout()
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    plt.savefig(tmp.name, dpi=150); plt.close()
    return tmp.name

# Gradio 버전에 따라 오디오가 tuple 또는 dict일 수 있어 표준화
def normalize_audio_item(item):
    if item is None:
        return None
    if isinstance(item, tuple) and len(item) == 2:
        sr, y = item
        return int(sr), np.asarray(y)
    if isinstance(item, dict):
        sr = item.get("sample_rate", item.get("sampling_rate"))
        y  = item.get("data")
        if sr is not None and y is not None:
            return int(sr), np.asarray(y)
    return None

# --- 라벨 & 분석 ---
labels = ["사과1","사과2","사과3","사과4","사과5"]

def analyze(a1, a2, a3, a4, a5, show_fft="사과1"):
    inputs = {"사과1": a1, "사과2": a2, "사과3": a3, "사과4": a4, "사과5": a5}
    feats, rows = {}, []
    fft_img = None

    for k in labels:
        item = normalize_audio_item(inputs[k])
        if item is None:
            rows.append([k, "-", "-", "-", "-", "-"])
            continue
        sr, y = item
        feat, freqs, mag = make_feature_from_wave(y, sr)
        feats[k] = feat
        rows.append([k] + [f"{x:.3f}" for x in feat])
        if k == show_fft:
            fft_img = plot_spectrum(freqs, mag)

    # 코사인 유사도 행렬
    sim_mat = np.zeros((len(labels), len(labels)), dtype=np.float32)
    for i, li in enumerate(labels):
        for j, lj in enumerate(labels):
            if (li in feats) and (lj in feats):
                sim_mat[i,j] = cosine_sim(feats[li], feats[lj])

    sim_img = plot_similarity_matrix(sim_mat, labels)
    return rows, sim_img, fft_img

# --- Gradio UI ---
with gr.Blocks() as demo:
    gr.Markdown("## 🎙️ 5명이 '사과'를 녹음 → **수학적 특징**으로 비교\n"
                "- 각자 마이크로 **'사과'**를 1~2초 말하고 **분석**을 누르세요.\n"
                "- 특징(RMS, ZCR, 저/중/고 대역 에너지) → **벡터**로 요약되어 **코사인 유사도**로 비교됩니다.\n")

    with gr.Row():
        with gr.Column():
            s1 = gr.Audio(sources=["microphone","upload"], type="numpy", label="사과1 (학생1)")
            s2 = gr.Audio(sources=["microphone","upload"], type="numpy", label="사과2 (학생2)")
            s3 = gr.Audio(sources=["microphone","upload"], type="numpy", label="사과3 (학생3)")
            s4 = gr.Audio(sources=["microphone","upload"], type="numpy", label="사과4 (학생4)")
            s5 = gr.Audio(sources=["microphone","upload"], type="numpy", label="사과5 (학생5)")
        with gr.Column():
            show_fft = gr.Dropdown(choices=labels, value="사과1", label="FFT 확인할 학생 선택")
            analyze_btn = gr.Button("분석")
            table = gr.Dataframe(headers=["label","RMS","ZCR","Low","Mid","High"],
                                 row_count=(5,"fixed"), col_count=(6,"fixed"))
            sim_plot = gr.Image(type="filepath", label="코사인 유사도 행렬")
            fft_plot = gr.Image(type="filepath", label="선택 학생 FFT 스펙트럼")
            analyze_btn.click(analyze, inputs=[s1,s2,s3,s4,s5,show_fft],
                              outputs=[table, sim_plot, fft_plot])

# Colab에서 마이크 권한 위해 HTTPS 공유 권장
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://32edd0fc1fe96c8f05.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
